### Why use the "multimodal only" dataset for a text-only project?



* **It filters out massive automated platform noise:** The "all text" dumps of Fakeddit include millions of blank posts, auto-generated bot threads, spam links, and deleted titles. The authors created the `multimodal_only` subset by isolating posts where a human user uploaded an image *and* wrote an original textual title. This process acts as a high-quality filter, leaving you with genuine, human-written content.
* **It preserves benchmark parity:** In the official Fakeddit paper, the authors explicitly state that their baseline experiments and error analyses were performed *strictly* on the multimodal samples. If you want to compare your BERT model's accuracy against the official published academic standard, you must train on the exact same data split they used.
* **Linguistic Richness:** Titles attached to images on Reddit (like memes, sensationalized news screenshots, or altered photos) are inherently richer in the specific types of manipulation you want to study—such as sarcasm, irony, misleading claims, and baiting language.

---

### Why not load test data now? 

Loading test data in exploratory phase would present a risk of data leakage.

* **Preventing Cognitive Bias:** If you compute class balances, check descriptive statistics, or look at vocabulary distributions on your test data, you might unconsciously write preprocessing rules or design choices tailored to those specific test samples. This compromises your model's validity.
* **Preserving the "Golden Standard":** Your test dataset must remain a completely unseen environment. Its only job is to sit quietly until the very end of your project. Once your model is fully trained on the `train` set and tuned using the `validate` set, you will pass the `test` data through it *once* to get your final thesis numbers.
* **Evaluation Strategy:** Eventually need to hand-label a tiny, custom *UK* test set to evaluate final system, the US Fakeddit test set is less critical for your immediate daily workflow. Your current focus is purely on building a working training pipeline.

---


1. **Train Set:** Used to let the BERT model learn patterns.
2. **Validation Set:** Used to check performance during training and tweak hyperparameters.
3. **Test Set:** untouched until evaluation


### 1.1 Read files

In [3]:
import pandas as pd
import os

data_dir = os.path.join('..', 'data', 'raw', 'fakeddit')

train_path = os.path.join(data_dir, 'multimodal_train.tsv')
val_path = os.path.join(data_dir, 'multimodal_validate.tsv')

train_fakeddit = pd.read_csv(train_path, sep='\t')
val_fakeddit = pd.read_csv(val_path, sep='\t')

In [4]:
train_fakeddit.head()

,author,clean_title,created_utc,domain,hasImage,id,image_url,linked_submission_id,num_comments,score,subreddit,title,upvote_ratio,2_way_label,3_way_label,6_way_label
0,Alexithymia,my walgreens offbrand mucinex was engraved wit...,1.551641e+09,i.imgur.com,True,awxhir,https://external-preview.redd.it/WylDbZrnbvZdB...,NaN,2.0,12,mildlyinteresting,My Walgreens offbrand Mucinex was engraved wit...,0.84,1,0,0
1,VIDCAs17,this concerned sink with a tiny hat,1.534727e+09,i.redd.it,True,98pbid,https://preview.redd.it/wsfx0gp0f5h11.jpg?widt...,NaN,2.0,119,pareidolia,This concerned sink with a tiny hat,0.99,0,2,2
2,prometheus1123,hackers leak emails from uae ambassador to us,1.496511e+09,aljazeera.com,True,6f2cy5,https://external-preview.redd.it/6fNhdbc6K1vFA...,NaN,1.0,44,neutralnews,Hackers leak emails from UAE ambassador to US,0.92,1,0,0
3,NaN,puppy taking in the view,1.471341e+09,i.imgur.com,True,4xypkv,https://external-preview.redd.it/HLtVNhTR6wtYt...,NaN,26.0,250,photoshopbattles,PsBattle: Puppy taking in the view,0.95,1,0,0
4,3rikR3ith,i found a face in my sheet music too,1.525318e+09,i.redd.it,True,8gnet9,https://preview.redd.it/ri7ut2wn8kv01.jpg?widt...,NaN,2.0,13,pareidolia,I found a face in my sheet music too!,0.84,0,2,2


### 1.2. Isolate relevant columns

In [12]:
rel_cols = ['clean_title', 'subreddit', 'domain', 'score', 'num_comments', 'upvote_ratio', 'created_utc', '2_way_label', '3_way_label', '6_way_label']
train_df = train_fakeddit[rel_cols].copy()
val_df = val_fakeddit[rel_cols].copy()

In [16]:
# Check for NaNs
print(train_fakeddit['clean_title'].isna().sum())
print(val_fakeddit['clean_title'].isna().sum())

print(train_fakeddit['score'].isna().sum())
print(train_fakeddit['num_comments'].isna().sum())
print(train_fakeddit['upvote_ratio'].isna().sum())

0
0
0
167857
167857


Fill NaN even if none present to avoid downstream issues

In [17]:
train_df['clean_title'] = train_df['clean_title'].fillna("")
val_df['clean_title'] = val_df['clean_title'].fillna("")

train_df['score'] = train_df['score'].fillna(0).astype(int)
train_df['num_comments'] = train_df['num_comments'].fillna(0).astype(int)
train_df['upvote_ratio'] = train_df['upvote_ratio'].fillna(1.0).astype(float)

### 1.3 Map labels

In [ ]:
fakeddit_6_way_labels = {
    0: "True",
    1: "Satire / Parody",
    2: "False Connection",
    3: "Imposter Content",
    4: "Manipulated Content",
    5: "Misleading Content"
}

train_df['label_name'] = train_df['6_way_label'].map(fakeddit_6_way_labels)
val_df['label_name'] = val_df['6_way_label'].map(fakeddit_6_way_labels)

In [19]:
train_df.head()

,clean_title,subreddit,domain,score,num_comments,upvote_ratio,created_utc,2_way_label,3_way_label,6_way_label,label_name
0,my walgreens offbrand mucinex was engraved wit...,mildlyinteresting,i.imgur.com,12,2,0.84,1.551641e+09,1,0,0,True / Authentic
1,this concerned sink with a tiny hat,pareidolia,i.redd.it,119,2,0.99,1.534727e+09,0,2,2,False Connection
2,hackers leak emails from uae ambassador to us,neutralnews,aljazeera.com,44,1,0.92,1.496511e+09,1,0,0,True / Authentic
3,puppy taking in the view,photoshopbattles,i.imgur.com,250,26,0.95,1.471341e+09,1,0,0,True / Authentic
4,i found a face in my sheet music too,pareidolia,i.redd.it,13,2,0.84,1.525318e+09,0,2,2,False Connection


### 2.1 Examine virality metrics

See how upvotes and comments vary across classes



In [ ]:
virality_summary = train_df.groupby('label_name')[['score', 'num_comments', 'upvote_ratio']].agg(['mean', 'median', 'max'])

virality_summary

--- Virality Metrics Across Misinformation Types ---


score                num_comments                \
                           mean median     max         mean median    max   
label_name                                                                  
False Connection     390.358773   17.0   93294     5.886636    1.0   3811   
Imposter Content     635.178717   18.0   50694    24.546419   20.0    314   
Manipulated Content  100.049620    7.0   15907     0.000000    0.0      0   
Misleading Content   372.963988   66.0   29059    20.637375    5.0   8832   
Satire / Parody      102.689466   11.0   34990     2.740719    1.0   5285   
True / Authentic     654.781255   15.0  137179    29.761249    3.0  10783   

                    upvote_ratio              
                            mean median  max  
label_name                                    
False Connection        0.872368   0.90  1.0  
Imposter Content        0.903087   0.93  1.0  
Manipulated Content     1.000000   1.00  1.0  
Misleading Content      0.928134   0.95  1.0  
Satire / Parody         0.892634   0.92  1.0  
True / Authentic        0.831929   0.85  1.0

### 2.2 Subreddit distribution

In [26]:
print("Top 10 Subreddits in Dataset")
print(train_df['subreddit'].value_counts().head(10))

# See how subreddits cross-reference with labels
print("\n--- Example: Subreddits making up 'Manipulated Content' (Label 4) ---")
print(train_df[train_df['6_way_label'] == 4]['subreddit'].value_counts().head(5))

Top 10 Subreddits in Dataset
subreddit
psbattle_artwork        167857
mildlyinteresting        86237
photoshopbattles         55198
pareidolia               47331
fakehistoryporn          35576
nottheonion              31977
upliftingnews            23960
fakealbumcovers          21725
misleadingthumbnails     15962
propagandaposters        13848
Name: count, dtype: int64

--- Example: Subreddits making up 'Manipulated Content' (Label 4) ---
subreddit
psbattle_artwork    167857
Name: count, dtype: int64
